# 생활권 군집화 — DBSCAN과 HDBSCAN, 적용하면 어떻게 되고 각각 무슨 이슈가 있나

> **3줄 결론**
> 1. 우리 데이터에 적용하면 **두 방식 모두 거의 같은 생활권**을 만듭니다. 누가 더 좋냐의 문제가 아닙니다.
> 2. **둘 다 이슈가 있습니다.** 종류가 다를 뿐입니다 — DBSCAN은 "반경(eps)을 정해야 하는" 이슈, HDBSCAN은 "왜 이렇게 묶였는지 설명할 수 없는" 이슈.
> 3. 우리는 규칙을 문장으로 쓸 수 있는 DBSCAN 변형을 쓰고, DBSCAN의 이슈(eps 근거)는 **k-거리 측정 절차**로 관리합니다. 지금의 260m는 문헌 대역(200~300m)에서 온 임시값이고, 운영값은 실데이터 측정으로 확정합니다. (§5에서 상세)

**사용법**: 메뉴 `런타임 → 모두 실행`. 코드는 접혀 있으니 그림과 글만 보면 됩니다.

---

## 용어 5개 (이것만 알면 됩니다)

| 용어 | 뜻 |
|---|---|
| **거점** | 반복해서 가는 곳 (집, 시장, 병원). 좌표 점들이 뭉쳐서 발견됨 |
| **eps** | "몇 미터 안에 모인 방문을 같은 장소로 볼 것인가"라는 반경. DBSCAN에만 있음 |
| **생활권** | 각 거점에 원(반경 = 최소 500m ~ 최대 2km)을 씌우고 **전부 합친 영역**. 판정은 이 합친 영역 기준 |
| **노이즈** | 어느 거점에도 못 묶여 버려진 방문 (한 번 가본 결혼식장 등) |
| **산포** | 같은 장소를 가도 주차 위치·GPS 오차 때문에 좌표가 흩어지는 정도 (보통 수십~수백 m) |


In [ ]:
#@title 준비 (데이터 다운로드 + 한글 폰트) — 실행만 하면 됩니다
import sys, os, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("seniorcareservice"):
        !git clone --branch claude/gaip-dashboard-refine --depth 1 -q https://github.com/summit1123/seniorcareservice.git
    ROOT = "seniorcareservice"
    !apt-get -qq -y install fonts-nanum > /dev/null 2>&1
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    FONT = "NanumGothic"
else:
    ROOT = ".."
    FONT = "Malgun Gothic"
sys.path.insert(0, ROOT)

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료")

---

## 0. 원리부터 — 어르신 한 분의 방문 12개로

알고리즘을 배우기 전에, 손으로 따라갈 수 있는 크기의 예제로 원리만 봅니다.
**김OO 어르신의 2주 방문 기록 12개**: 집 6번(월~토), 시장 3번(화·목·토), 병원 2번(수·금), 결혼식장 1번(일).

**DBSCAN의 규칙은 문장 하나입니다** — "반경 260m 안에 서로 다른 3일 이상 방문이 모이면 거점."
- 집: 6일 → 거점 ✓ · 시장: 3일 → 거점 ✓ · 병원: 2일 → 미달, 노이즈 × · 결혼식장: 1일 → 노이즈 ×

**HDBSCAN은 반경을 정하지 않습니다.** 대신 반경을 0부터 계속 키우면서 뭉침 과정 전체를 지켜봅니다 —
수십 m에서 집·시장·병원이 각자 뭉치고, 570m부터 서로 다른 장소끼리 합쳐지기 시작합니다.
그중 "넓은 반경 구간 동안 오래 따로 유지된" 묶음(집, 시장)을 군집으로 선택하고,
점 2개뿐인 병원은 최소 크기 미달로 노이즈가 됩니다.

아래 그림이 이 과정입니다. **왼쪽 = DBSCAN의 답, 가운데 = 반경을 키우는 과정의 한 장면, 오른쪽 = 너무 키웠을 때.**

In [ ]:
#@title 그림 0 — 원리 미니 예제 (방문 12개)
import math as _m

MINI = [
    (10, 20, "월", "집"), (-30, -10, "화", "집"), (25, -35, "수", "집"),
    (-15, 40, "목", "집"), (40, 5, "금", "집"), (-40, -30, "토", "집"),
    (590, 140, "화", "시장"), (620, 165, "목", "시장"), (605, 130, "토", "시장"),
    (340, -445, "수", "병원"), (365, -460, "금", "병원"),
    (-650, 380, "일", "결혼식장"),
]
PLACE_COLOR = {"집": "#1D9E75", "시장": "#378ADD"}
ANCHORS = {"집": (0, 0), "시장": (605, 145), "병원": (352, -452), "결혼식장": (-650, 380)}

def draw_points(ax, color_by_place=True, noise_places=("병원", "결혼식장"), one_color=None):
    for (x, y, d, p) in MINI:
        if one_color:
            ax.scatter(x, y, c=one_color, s=45, zorder=3)
        elif color_by_place and p in PLACE_COLOR:
            ax.scatter(x, y, c=PLACE_COLOR[p], s=45, zorder=3)
        elif p in noise_places:
            ax.scatter(x, y, marker="x", c="#999999", s=60, zorder=3)
        else:
            ax.scatter(x, y, c="#888780", s=45, zorder=3)
    for name, (x, y) in ANCHORS.items():
        ax.annotate(name, (x, y), textcoords="offset points", xytext=(0, 26),
                    ha="center", fontsize=11)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
ax = axes[0]
draw_points(ax)
for name, r, c in [("집", 500, "#1D9E75"), ("시장", 500, "#378ADD")]:
    x, y = ANCHORS[name]
    ax.add_patch(plt.Circle((x, y), r, fill=True, alpha=0.08, color=c))
    ax.add_patch(plt.Circle((x, y), r, fill=False, linestyle="--", color=c))
ax.set_title("DBSCAN의 답 — 3일 충족한 집·시장만 거점\n병원(2일)·결혼식장(1일)은 노이즈(×)", fontsize=11)

ax = axes[1]
draw_points(ax, color_by_place=True, noise_places=())
for name in ("집", "시장", "병원"):
    x, y = ANCHORS[name]
    ax.add_patch(plt.Circle((x, y), 90, fill=False, linestyle=":", color="#888780"))
ax.set_title("반경을 키우는 과정(약 90m 장면)\n각 장소가 먼저 각자 뭉친다", fontsize=11)

ax = axes[2]
draw_points(ax, one_color="#D4537E")
ax.add_patch(plt.Circle((60, -60), 780, fill=False, linestyle="--", color="#D4537E"))
ax.set_title("반경 700m+ — 전부 한 덩어리로 병합\n'어디서 멈추나'가 곧 알고리즘의 답", fontsize=11)

for ax in axes:
    ax.set_aspect("equal"); ax.grid(alpha=0.2)
    ax.set_xlim(-900, 900); ax.set_ylim(-750, 700)
fig.tight_layout(); plt.show()

### 0.5 같은 내용을 수식으로 — 두 알고리즘 대조

**"이웃"이라는 말부터**: 이웃 = 어떤 방문점 주변, 일정 거리 안에 있는 다른 방문점들.
두 알고리즘의 차이는 그 "일정 거리"를 **누가 정하느냐** 하나뿐입니다.

| | DBSCAN (우리 변형) | HDBSCAN |
|---|---|---|
| 이웃의 반경 | **사람이 정함**: $\varepsilon$ = 260m | **데이터가 정함**: 점마다 핵심거리 $\mathrm{core}_k(p)$ 를 계산 |
| 이웃 정의 | $N(p) = \{\, q : d(p,q) \le \varepsilon \,\}$ | 고정 반경 없음 — 아래 연결 반경으로 대체 |
| 거점 판정 | $N(p)$ 안의 **서로 다른 방문일** $\ge 3$ 이면 $p$는 거점의 씨앗 | 두 점의 연결 반경 $d_{mr}(p,q)=\max(\mathrm{core}_k(p),\,\mathrm{core}_k(q),\,d(p,q))$ 로 뭉침 트리를 만들고, **오래 유지된** 묶음만 군집으로 선택 |
| 군집 만들기 | 씨앗에서 이웃을 따라 연결된 점 전부 | 트리에서 선택된 묶음 (크기 $\ge$ min_cluster_size) |

기호 읽는 법: $p, q$ = 방문점 하나씩 · $d(p,q)$ = 두 점 사이 거리(미터, 곡면 보정) · $\varepsilon$(eps) = 한 장소로 묶는 반경 · $\mathrm{core}_k(p)$ = $p$에서 $k$번째로 가까운 점까지의 거리

**핵심거리를 숫자로**: 점 A에서 가장 가까운 방문이 100m, 그다음이 500m라면 → $\mathrm{core}_2(A) = 500$m.
"A 주변은 500m는 나가야 방문 2개가 모이는 한산한 동네"라는 뜻입니다. 그래서 A는 100m 옆 점과도
반경이 500m로 커져야 연결됩니다 — 한산한 동네의 우연한 몰림을 걸러내는 장치입니다.

**파라미터 전체 목록** (누가 정하고, 근거가 뭔가):

| 파라미터 | 어느 쪽 | 뜻 | 우리 값 | 근거 |
|---|---|---|---|---|
| $\varepsilon$ (eps) | DBSCAN | 한 장소로 묶는 반경 | 260m | 문헌 대역 200~300m (임시) → 실데이터 k-거리로 확정 예정 |
| min_days | DBSCAN 변형 | 거점 인정에 필요한 서로 다른 방문일 | 3일 | 상품 규칙 그대로 (하루 몰림 방지) |
| min_cluster_size | HDBSCAN | 거점 후보가 되는 최소 점 개수 | 5 (실험) | 근거 문헌 없음 — 민감 (이슈 H2) |
| min_samples ($k$) | HDBSCAN | 핵심거리의 $k$ = 밀집 판단의 보수성 | 3 (실험) | 근거 문헌 없음 — 민감 (이슈 H2) |

In [ ]:
#@title 공통 코드 (데이터 로드 + 군집/생활권 계산 함수) — 실행만 하면 됩니다
import csv, math, random
import numpy as np
from collections import defaultdict
from src.gaip_simulation.clustering import dbscan_distinct_days, haversine_m, percentile_nearest_rank

CSV_PATH = ROOT + "/data/fixtures/gaip_visit_events.csv"
CORE_M, CAP_M = 500.0, 2000.0
COLORS = ["#1D9E75", "#378ADD", "#EF9F27", "#D4537E", "#7F77DD", "#639922", "#D85A30", "#0F6E56"]

with open(CSV_PATH, encoding="utf-8") as f:
    ALL_ROWS = list(csv.DictReader(f))
for r in ALL_ROWS:
    r["latitude"] = float(r["latitude"]); r["longitude"] = float(r["longitude"])

def load_driver(driver_id):
    rows = [r for r in ALL_ROWS if r["driver_id"] == driver_id]
    fit = [r for r in rows if r["period_role"] == "baseline"]
    ev = [r for r in rows if r["period_role"] != "baseline"]
    return fit, ev

def offsets(events, anchor):
    lat0, lon0 = anchor
    return [((e["longitude"] - lon0) * 111_320.0 * math.cos(math.radians(lat0)),
             (e["latitude"] - lat0) * 111_320.0) for e in events]

def zone_clusters(events, labels):
    groups = defaultdict(list)
    for e, l in zip(events, labels):
        if l >= 0:
            groups[l].append(e)
    out = []
    for l, evs in sorted(groups.items()):
        lat = sum(e["latitude"] for e in evs) / len(evs)
        lon = sum(e["longitude"] for e in evs) / len(evs)
        d = [haversine_m(e["latitude"], e["longitude"], lat, lon) for e in evs]
        p90 = percentile_nearest_rank(d, 0.90)
        out.append({"lat": lat, "lon": lon, "r": max(CORE_M, min(p90, CAP_M))})
    return out

def coverage(eval_events, clusters):
    if not clusters or not eval_events:
        return 0.0
    inz = sum(1 for e in eval_events
              if any(haversine_m(e["latitude"], e["longitude"], c["lat"], c["lon"]) <= c["r"]
                     for c in clusters))
    return inz / len(eval_events) * 100

def run_hdbscan(events, anchor, mcs, ms):
    from sklearn.cluster import HDBSCAN
    xy = offsets(events, anchor)
    h = HDBSCAN(min_cluster_size=mcs, min_samples=ms, allow_single_cluster=True, copy=True)
    return h.fit_predict(np.array(xy)).tolist()

# 본문 그림은 광역(저밀도) 지역의 다생활권 운전자 1명으로 통일
FIT, EV = load_driver("gaip-123")
ANCHOR = (sum(e["latitude"] for e in FIT) / len(FIT), sum(e["longitude"] for e in FIT) / len(FIT))

def panel(ax, labels, title):
    clusters = zone_clusters(FIT, labels)
    cov = coverage(EV, clusters)
    for (x, y), l in zip(offsets(FIT, ANCHOR), labels):
        if l < 0:
            ax.scatter(x, y, marker="x", c="#999999", s=42, zorder=3)
        else:
            ax.scatter(x, y, c=COLORS[l % len(COLORS)], s=30, zorder=3, edgecolors="none")
    for i, c in enumerate(clusters):
        cx = (c["lon"] - ANCHOR[1]) * 111_320.0 * math.cos(math.radians(ANCHOR[0]))
        cy = (c["lat"] - ANCHOR[0]) * 111_320.0
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=True, alpha=0.10,
                                color=COLORS[i % len(COLORS)], zorder=1))
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=False, linestyle="--", linewidth=1.4,
                                color=COLORS[i % len(COLORS)], zorder=2))
    noise = sum(1 for l in labels if l < 0)
    ax.set_title(f"{title}\n거점 {len(clusters)}곳 · 버려진 방문 {noise}건 · 생활권이 담은 방문 {cov:.0f}%",
                 fontsize=11)
    ax.set_aspect("equal"); ax.grid(alpha=0.25)
    ax.set_xlabel("동쪽 (m)"); ax.set_ylabel("북쪽 (m)")
print("공통 코드 준비 완료")

---

## 1. 두 방식 모두, 적용되는 방식은 같습니다

어느 알고리즘을 쓰든 파이프라인은 동일합니다:

```text
방문 좌표(기준선 2개월) → [군집화: 뭉친 곳 찾기] → 거점마다 원 씌우기 → 원 합집합 = 생활권 → 이후 주행을 안/밖 판정
```

알고리즘이 바꾸는 건 가운데 **[뭉친 곳 찾기]** 한 단계뿐입니다.
아래는 실제 데이터의 운전자 한 명(거점 3곳을 오가는 저밀도 지역 거주자)에게 이 파이프라인이 적용된 모습입니다.

In [ ]:
#@title 그림 1 — 생활권이 만들어지는 과정 (기본 적용)
labels = dbscan_distinct_days(FIT, eps_m=260, min_distinct_days=3)["labels"]
fig, ax = plt.subplots(figsize=(7.5, 8))
panel(ax, labels, "방문 점이 뭉친 곳 = 거점 · 점선 원을 합친 영역 = 생활권")
fig.tight_layout(); plt.show()

---

## 2. DBSCAN을 적용하면 — 그리고 DBSCAN의 이슈

**적용 방식 (우리 변형)**: 아래 한 문장이 알고리즘의 전부입니다.

> "반경 **eps(260m)** 안에, **서로 다른 3일 이상** 방문이 모이면 거점으로 인정한다."

'서로 다른 3일'은 우리가 표준 DBSCAN을 고친 부분입니다 — 표준은 "점 3개"라서 **하루에 10번 정차한 곳(이삿날, 행사)도 거점으로 오인**하는데, 우리 규칙은 날짜로 세서 이걸 막습니다. 상품 규칙이 알고리즘 안에 그대로 들어간 구조라 약관·심사 문장으로 쓸 수 있습니다.

### DBSCAN의 이슈: eps라는 숫자를 정해야 한다

이슈는 하나로 모입니다 — **eps를 정해야 하고, 잘못 정하면 각 방향으로 대가가 있습니다.**

| eps를 | 무슨 일이 생기나 | 실측 |
|---|---|---|
| 너무 크게 (예: 1,100m) | 서로 다른 거점이 한 덩어리로 **병합** → 원이 상한(2km)에 걸려 생활권이 실제보다 덜 덮임 | 아래 그림 2 오른쪽: 담은 방문 100% → 96% |
| 너무 작게 (산포보다 작게) | 산포가 큰 동네에서 거점이 조각나고 방문이 버려짐 → 극단적으론 생활권 미형성 | 산포를 400m로 키운 실험에서 방문 20% 버림, 생활권 미형성 1명 발생 (부록 C) |

그리고 "왜 260m인가?"라는 **근거 질문**을 받게 됩니다. 이 근거를 만드는 절차가 아래 4번(k-거리)입니다.

In [ ]:
#@title 그림 2 — DBSCAN의 이슈: eps를 잘못 정하면 (같은 사람, eps만 다르게)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
labels_260 = dbscan_distinct_days(FIT, eps_m=260, min_distinct_days=3)["labels"]
labels_1100 = dbscan_distinct_days(FIT, eps_m=1100, min_distinct_days=3)["labels"]
panel(axes[0], labels_260, "eps=260m — 거점 3곳이 각자 인정됨 (정상)")
panel(axes[1], labels_1100, "eps=1,100m — 남쪽 두 거점이 병합됨 (이슈)")
fig.tight_layout(); plt.show()

---

## 3. HDBSCAN을 적용하면 — 그리고 HDBSCAN의 이슈

**적용 방식**: eps를 주지 않습니다. 대신 알고리즘이 **그 사람 점들의 흩어진 정도를 보고 묶는 반경을 스스로 정합니다.**
사람이 정하는 건 두 개 — `min_cluster_size`(몇 개짜리 뭉치부터 거점 후보로 보나), `min_samples`(밀집 판단을 얼마나 보수적으로 하나).

잘 맞추면 결과는 DBSCAN과 사실상 같습니다 (아래 그림 3 왼쪽). **즉 "성능이 나빠서" 탈락한 게 아닙니다.**

### HDBSCAN의 이슈 4가지

| # | 이슈 | 왜 문제인가 |
|---|---|---|
| H1 | **왜 이렇게 묶였는지 문장으로 설명 불가** | 묶음 기준이 "여러 반경에 걸쳐 오래 살아남은 덩어리"라는 내부 계산이라, 약관·민원 답변·감사 문장으로 쓸 수 없음. "고객님의 거점은 반경 260m 안 3일 이상 방문 기준입니다" 같은 문장이 불가능 |
| H2 | **설정 2개에 결과가 민감** | 아래 그림 3: 설정을 5/3 → 3/2로 바꾸면 같은 사람의 거점이 3곳 → 6조각. 우리 180명 실측에서도 거점 쪼개짐 비율이 12% ↔ 47%로 출렁임. scikit-learn 공식 문서도 이 민감성을 명시함 → "eps 근거 질문"이 "min_cluster_size 근거 질문"으로 바뀔 뿐 |
| H3 | **적은 데이터에서 우연을 구조로 오인** | 개인당 점 ~100개뿐이라, 우연한 주차 몰림 두 군데를 "서로 다른 거점 2개"로 읽을 수 있음 (단, 원 합집합이 대부분 흡수 — 아래 4번 공통 이슈 참고) |
| H4 | **사전에 기준을 고정할 수 없음** | "몇 미터까지 같은 장소로 묶는다"를 미리 문서에 적을 수 없음 — 보험 상품은 기준을 먼저 고정하고 적용해야 하는데 순서가 반대가 됨 |

In [ ]:
#@title 그림 3 — HDBSCAN의 이슈: 설정에 따라 결과가 출렁임 (같은 사람, 설정만 다르게)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
labels_53 = run_hdbscan(FIT, ANCHOR, 5, 3)
labels_32 = run_hdbscan(FIT, ANCHOR, 3, 2)
panel(axes[0], labels_53, "설정 5/3 — DBSCAN과 사실상 같은 결과")
panel(axes[1], labels_32, "설정 3/2 — 거점이 잘게 쪼개짐 (단, 원 합집합은 유지)")
fig.tight_layout(); plt.show()

---

## 3.5 직접 바꿔보기 — 우리 데이터에서 파라미터 움직여보기

아래 셀 상단의 값 4개를 바꾸고 셀을 다시 실행하면(셀 클릭 후 Shift+Enter), 다른 운전자·다른 설정의
결과를 바로 볼 수 있습니다. 표의 대표 운전자 18명(유형 6 × 지역 3)이 먼저 출력됩니다.

In [ ]:
DRIVER_ID = "gaip-123"   # 아래 목록에서 골라 바꿔보세요
EPS_M = 260              # DBSCAN: 한 장소로 묶는 반경(m) — 100~1200 사이로 바꿔보세요
MIN_DAYS = 3             # DBSCAN: 거점 인정에 필요한 서로 다른 방문일
MCS, MS = 5, 3           # HDBSCAN: (min_cluster_size, min_samples) — 3,2로 바꾸면 조각남

reps = {}
for r in ALL_ROWS:
    reps.setdefault((r["environment_id"], r["designed_type"]), r["driver_id"])
print(f"{'지역':22s} {'유형':24s} 대표 운전자")
for (env, t), d in sorted(reps.items()):
    print(f"{env:22s} {t:24s} {d}")

fit_x, ev_x = load_driver(DRIVER_ID)
anchor_x = (sum(e["latitude"] for e in fit_x) / len(fit_x),
            sum(e["longitude"] for e in fit_x) / len(fit_x))

def panel_x(ax, labels, title):
    clusters = zone_clusters(fit_x, labels)
    cov = coverage(ev_x, clusters)
    for (x, y), l in zip(offsets(fit_x, anchor_x), labels):
        if l < 0:
            ax.scatter(x, y, marker="x", c="#999999", s=42, zorder=3)
        else:
            ax.scatter(x, y, c=COLORS[l % len(COLORS)], s=30, zorder=3, edgecolors="none")
    for i, c in enumerate(clusters):
        cx = (c["lon"] - anchor_x[1]) * 111_320.0 * math.cos(math.radians(anchor_x[0]))
        cy = (c["lat"] - anchor_x[0]) * 111_320.0
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=True, alpha=0.10,
                                color=COLORS[i % len(COLORS)], zorder=1))
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=False, linestyle="--", linewidth=1.4,
                                color=COLORS[i % len(COLORS)], zorder=2))
    noise = sum(1 for l in labels if l < 0)
    ax.set_title(f"{title}\n거점 {len(clusters)}곳 · 버려진 방문 {noise}건 · 담은 방문 {cov:.0f}%", fontsize=11)
    ax.set_aspect("equal"); ax.grid(alpha=0.25)
    ax.set_xlabel("동쪽 (m)"); ax.set_ylabel("북쪽 (m)")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
panel_x(axes[0], dbscan_distinct_days(fit_x, eps_m=EPS_M, min_distinct_days=MIN_DAYS)["labels"],
        f"DBSCAN eps={EPS_M}m + {MIN_DAYS}일 — {DRIVER_ID}")
panel_x(axes[1], run_hdbscan(fit_x, anchor_x, MCS, MS),
        f"HDBSCAN {MCS}/{MS} — {DRIVER_ID}")
fig.tight_layout(); plt.show()

---

## 4. 둘 다 갖고 있는 공통 이슈 (알고리즘 탓이 아닌 것)

어느 쪽을 쓰든 그대로 남는 이슈들입니다. 알고리즘 논쟁과 분리해서 관리해야 합니다.

| # | 공통 이슈 | 실측 |
|---|---|---|
| C1 | **원 반경 상한(2km)이 아주 넓은 생활권을 다 못 덮음** | 저밀도(광역) 지역 커버리지가 다른 지역보다 낮음 (86% vs 91%) — 알고리즘이 아니라 원 규칙의 상한 문제 |
| C2 | **최소 반경 500m가 커서 새 목적지 방문을 기존 생활권이 흡수** | 새 목적지 방문의 62%가 기존 존 안으로 들어옴 — 어느 알고리즘이든 동일. 변화 감지 민감도에 영향 |
| C3 | **합성 데이터 검증의 한계** | 지금까지의 모든 수치는 우리가 만든 데이터 기준. 실데이터에서 재측정 전에는 어느 쪽도 "검증됐다"고 말할 수 없음 |

**중요한 발견 하나**: 거점이 몇 조각으로 쪼개져도(H3), 조각마다 원을 씌워 합치면 생활권 영역은 거의 같아집니다.
그림 3 오른쪽에서 거점이 6조각인데도 "담은 방문"이 유지되는 이유입니다. **쪼개짐 자체는 상품 판정에 거의 해가 없습니다** — 이건 두 알고리즘 모두에게 적용되는 완충 장치입니다.

---

## 5. 그래서 260m는 어디서 왔나 — 그리고 "k-거리"가 뭔가

### 먼저: 지금 쓰는 260m의 출처 (합성 데이터 아님)

지금의 260m는 **문헌에서 온 값**입니다. 모빌리티 연구에서 "한 장소"를 판정하는 공간 반경이
**200~300m 대역**에 몰려 있습니다:

| 선례 | 값 | 출처 |
|---|---|---|
| GPS 궤적에서 체류지(stay point) 판정 반경 | **200m** | Zheng et al., *Mining Interesting Locations and Travel Sequences from GPS Trajectories*, WWW 2009 (Microsoft GeoLife) |
| 스마트폰 데이터의 관심장소 반경 | **250m** (권장 200~300m) | Montoliu et al., *Discovering Places of Interest in Everyday Life from Smartphone Data*, 2013 |
| "서로 다른 3일" 일수 기준의 직접 선례 | 60일 관찰 · 전체 일수의 5% = **3일** | Isaacman et al., *Identifying Important Places in People's Lives from Cellular Network Data*, Pervasive 2011 — [원문 PDF](https://mrmgroup.cs.princeton.edu/papers/Isaacman_pervasive11.pdf) |
| "전역 고정 eps는 밀도 다른 군집을 병합할 수 있다"(이슈 D1의 원출처) | — | Ester et al., DBSCAN 원논문, KDD 1996, p.229 — [원문 PDF](http://cdn.aaai.org/KDD/1996/KDD96-037.pdf) |

260m는 이 공간 임계 대역(200~300m)의 가운데를 임시 운영값으로 채택한 것입니다.
푸아송 오류율 계산·국내 방문빈도 통계(건보공단·노인실태조사) 등 전체 근거 스택은
저장소 [final 폴더의 근거 노트](https://github.com/summit1123/seniorcareservice/tree/claude/gaip-dashboard-refine/final)
(`EVIDENCE_기준선_방문임계_노트.md`)에 원문 인용으로 정리돼 있습니다. 정리하면 2단 구조입니다:

> **지금: 문헌 대역(200~300m) 기반 임시값 → 운영: 실데이터 측정(k-거리)으로 확정.**

그 "실데이터 측정"을 담당하는 절차가 k-거리입니다. 풀어서:

**k-거리 = 어떤 방문점에서, "다른 날"의 방문점 중 k번째로 가까운 것까지의 거리.**

우리 세팅의 기준 3가지와 그 이유:

| 기준 | 값 | 왜 이 값인가 |
|---|---|---|
| k | **2** | 거점 인정 규칙이 "서로 다른 3일"이므로, 어떤 점이 거점 후보가 되려면 자기 날 말고 **다른 날 이웃이 2개** 필요 → "2번째 다른-날 이웃까지의 거리"가 곧 "같은 장소 재방문이 흩어지는 폭"의 측정치 |
| 다른 날만 셈 | — | 같은 날 여러 번 정차는 재방문이 아니라 한 외출이므로 제외 (3일 규칙과 같은 철학) |
| 컷 | **P95** (지역별로 모은 k-거리 분포의 95% 지점) | "재방문 쌍의 95%를 같은 장소로 인정하는 반경"이라는 뜻. 꼬리 5%는 이상치로 배제 — 존 반경에 P90을 쓰는 것과 같은 철학. ※ 이 백분위(95)가 사람이 정하는 마지막 손잡이라는 건 인정하고 가야 함 |

이 절차가 실데이터에 적용되면 "왜 260m인가?"의 답이 **"문헌에서 골랐다"에서 "재방문 산포를 재면 이 값이 나온다"로 업그레이드**됩니다.

**아래 표를 오해하지 마세요**: 합성 데이터에 절차를 돌려본 **리허설**이며, 나오는 숫자(30~80m)는 합성 산포일 뿐
**260의 근거가 아닙니다** (합성 데이터로는 260을 정당화할 수도, 반박할 수도 없습니다 — 우리가 만든 세계라서요).
이 표의 확인 포인트는 하나: 지역별로 산포가 다르면 절차가 그 차이를 잡아낸다는 것.
절차가 **진짜 데이터에서** 작동하는지는 부록 D(실제 GPS)에서 확인합니다.

In [ ]:
#@title 표 — k-거리 측정 실행 (지역별 재방문 산포)
by_env_driver = defaultdict(lambda: defaultdict(list))
for r in ALL_ROWS:
    if r["period_role"] == "baseline":
        by_env_driver[r["environment_id"]][r["driver_id"]].append(
            (r["latitude"], r["longitude"], r["visit_date"]))

def pct(xs, q):
    xs = sorted(xs)
    return xs[max(0, min(len(xs) - 1, int(len(xs) * q)))]

ENV_KO = {"dense_urban": "도심", "suburban_mid_density": "교외", "wide_low_density": "광역(저밀도)"}
print(f"{'지역':12s} {'중앙값':>7s} {'P90':>7s} {'P95':>7s}   -> 이 지역 eps로 쓸 값")
for env, drivers in by_env_driver.items():
    pooled = []
    for d, pts in drivers.items():
        for i, (la, lo, dt) in enumerate(pts):
            ds = sorted(haversine_m(la, lo, la2, lo2)
                        for j, (la2, lo2, dt2) in enumerate(pts) if j != i and dt2 != dt)
            if len(ds) >= 2:
                pooled.append(ds[1])
    p50, p90, p95 = pct(pooled, 0.50), pct(pooled, 0.90), pct(pooled, 0.95)
    print(f"{ENV_KO[env]:12s} {p50:6.0f}m {p90:6.0f}m {p95:6.0f}m   -> ~{math.ceil(p95/10)*10}m")
print()
print("주의: 합성 데이터의 산포는 실제(GPS 오차 10~30m + 주차장 크기)보다 훨씬 작습니다.")
print("실데이터 파일럿에서 같은 절차로 다시 재서 운영 eps를 확정합니다.")

---

## 6. 최종 정리 — 이슈 프로필 비교와 우리 선택

| | DBSCAN (우리 변형) | HDBSCAN |
|---|---|---|
| 적용하면 | 문장 하나짜리 규칙으로 거점 인정 | 알고리즘이 반경을 스스로 정해 거점 인정 |
| 우리 데이터 결과 | 생활권 거의 동일 | 생활권 거의 동일 |
| 고유 이슈 | eps를 정해야 함 (크면 병합, 작으면 산포 큰 동네서 조각·유실) | 설명 문장화 불가 · 설정 2개 민감 · 소표본 오인 · 기준 사전 고정 불가 |
| 이슈 관리 방법 | **k-거리 측정으로 eps를 "정하지 않고 잰다"** + 채점표 상시 검증 | 민감도 검증 필요 + 설명 불가는 관리 불가능(구조적) |
| 공통 이슈 | 원 상한 2km · 최소 반경 500m의 변화 흡수 · 합성 검증 한계 — **알고리즘 교체로 해결 안 됨** | (동일) |

**우리 선택과 그 이유**: 성능이 같다면 선택 기준은 이슈의 관리 가능성입니다.
DBSCAN의 이슈(eps)는 측정 절차로 관리가 **되고**, HDBSCAN의 핵심 이슈(설명 불가)는 관리가 **안 됩니다**(알고리즘의 구조라서).
그래서 현행 = DBSCAN 변형 + k-거리 절차이고, HDBSCAN은 실데이터 파일럿에서 같은 채점표로 나란히 재평가합니다.
단, 실측 산포가 eps급으로 큰 지역이 확인되면(부록 C의 조건) 그 지역은 eps 상향 또는 HDBSCAN 전환을 검토합니다.

---

# 부록 — 더 깊이 보고 싶은 사람용

이하는 본문 근거의 상세입니다. 안 봐도 본문 이해에는 지장 없습니다.

- **부록 A**: scikit-learn 공식 DBSCAN 데모 (원본 코드)
- **부록 B**: scikit-learn 공식 HDBSCAN 데모 (원본 코드) — 스케일 민감성 · 밀도 혼합 · 설정 민감도
- **부록 C**: 산포 스트레스 테스트 — 두 알고리즘이 갈라지는 조건의 실측


## 부록 A — 공식 DBSCAN 데모 (원본)

출처: https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html (BSD-3-Clause)

In [ ]:
# 출처: scikit-learn 공식 예제 "Demo of DBSCAN clustering algorithm" (원본)
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0
)

X = StandardScaler().fit_transform(X)

from sklearn import metrics
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.3, min_samples=10).fit(X)
labels = db.labels_

n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)
print(f"Silhouette Coefficient: {metrics.silhouette_score(X, labels):.3f}")

unique_labels = set(labels)
core_samples_mask = np.zeros_like(labels, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True

colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]

    class_member_mask = labels == k

    xy = X[class_member_mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=14)

    xy = X[class_member_mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=6)

plt.title(f"Estimated number of clusters: {n_clusters_}")
plt.show()

## 부록 B — 공식 HDBSCAN 데모 (원본)

출처: https://scikit-learn.org/stable/auto_examples/cluster/plot_hdbscan.html (BSD-3-Clause)

세 장면: ① 축척만 바꿔도 같은 eps가 무너짐(스케일 민감성) ② 밀도가 다른 무리가 한 판에 있으면
어떤 eps도 동시에 만족 못 함(HDBSCAN의 존재 이유) ③ HDBSCAN도 min_cluster_size/min_samples에 민감(본문 이슈 H2의 출처)

In [ ]:
# 출처: scikit-learn 공식 예제 "Demo of HDBSCAN clustering algorithm" (원본)
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause
from sklearn.cluster import HDBSCAN
from sklearn.datasets import make_blobs


def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    labels = labels if labels is not None else np.ones(X.shape[0])
    probabilities = probabilities if probabilities is not None else np.ones(X.shape[0])
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    proba_map = {idx: probabilities[idx] for idx in range(len(labels))}
    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = [0, 0, 0, 1]

        class_index = (labels == k).nonzero()[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0],
                X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=4 if k == -1 else 1 + 5 * proba_map[ci],
            )
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" | {parameters_str}"
    ax.set_title(title)
    plt.tight_layout()


# ① 스케일 민감성
centers = [[1, 1], [-1, -1], [1.5, -1.5]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.4, 0.1, 0.75], random_state=0
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
dbs = DBSCAN(eps=0.3)
for idx, scale in enumerate([1, 0.5, 3]):
    dbs.fit(X * scale)
    plot(X * scale, dbs.labels_, parameters={"scale": scale, "eps": 0.3}, ax=axes[idx])

fig, axes = plt.subplots(3, 1, figsize=(10, 12))
hdb = HDBSCAN(copy=True)
for idx, scale in enumerate([1, 0.5, 3]):
    hdb.fit(X * scale)
    plot(X * scale, hdb.labels_, hdb.probabilities_, ax=axes[idx],
         parameters={"scale": scale})

In [ ]:
# ② 밀도 혼합 (멀티스케일) — 원본 그대로
centers = [[-0.85, -0.85], [-0.85, 0.85], [3, 3], [3, -3]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.2, 0.35, 1.35, 1.35], random_state=0
)
plot(X, labels=labels_true, ground_truth=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
params = {"eps": 0.7}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[0])
params = {"eps": 0.3}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[1])

hdb = HDBSCAN(copy=True).fit(X)
plot(X, hdb.labels_, hdb.probabilities_)

In [ ]:
# ③ HDBSCAN 하이퍼파라미터 민감도 — 원본 그대로
PARAM = ({"min_cluster_size": 5}, {"min_cluster_size": 3}, {"min_cluster_size": 25})
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

PARAM = (
    {"min_cluster_size": 20, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 3},
    {"min_cluster_size": 20, "min_samples": 25},
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

## 부록 C — 산포 스트레스 테스트 (두 알고리즘이 갈라지는 조건)

주차 산포를 인위적으로 키우며 광역 60명을 재실행한 결과입니다. 산포가 eps(260m)를 넘는
순간부터 두 알고리즘이 갈라집니다 — 본문 6번 "eps 상향/전환 검토 조건"의 근거.

In [ ]:
def jitter(events, sigma, rng):
    out = []
    for e in events:
        out.append({"latitude": e["latitude"] + rng.gauss(0, sigma) / 111_320.0,
                    "longitude": e["longitude"] + rng.gauss(0, sigma)
                    / (111_320.0 * math.cos(math.radians(e["latitude"]))),
                    "visit_date": e["visit_date"]})
    return out

wide = defaultdict(lambda: {"fit": [], "ev": []})
for r in ALL_ROWS:
    if r["environment_id"] != "wide_low_density":
        continue
    e = {"latitude": r["latitude"], "longitude": r["longitude"], "visit_date": r["visit_date"]}
    wide[r["driver_id"]]["fit" if r["period_role"] == "baseline" else "ev"].append(e)

print(f"{'산포':>6s} | {'방식':16s} | {'생활권 미형성':>8s} | {'평균 거점':>6s} | {'버려진 방문':>7s} | {'담은 방문':>7s}")
for sigma in [0, 200, 400]:
    for name, fn in [("DBSCAN 260m/3일",
                      lambda f: dbscan_distinct_days(f, eps_m=260, min_distinct_days=3)["labels"]),
                     ("HDBSCAN 5/3",
                      lambda f: run_hdbscan(f, (sum(e['latitude'] for e in f) / len(f),
                                                sum(e['longitude'] for e in f) / len(f)), 5, 3))]:
        rng2 = random.Random(42)
        zero = 0; hubs = []; noise = 0; pts = 0; covs = []
        for d, dd in wide.items():
            fitj = jitter(dd["fit"], sigma, rng2)
            evj = jitter(dd["ev"], sigma, rng2)
            labels = fn(fitj)
            zs = zone_clusters(fitj, labels)
            if not zs:
                zero += 1
            hubs.append(len(zs)); noise += sum(1 for l in labels if l < 0); pts += len(labels)
            covs.append(coverage(evj, zs))
        print(f"{sigma:5d}m | {name:16s} | {zero:6d}명 | {sum(hubs)/len(hubs):8.1f} "
              f"| {noise/pts*100:6.1f}% | {sum(covs)/len(covs):6.1f}%")

## 부록 D — 진짜 GPS 데이터로 직접 확인 (합성 아님)

**이 부록이 있는 이유**: 본문의 모든 수치는 우리가 만든 합성 데이터 기준이라,
"합성으로 만들고 합성으로 검증했다"는 순환에 갇힙니다. 그 순환을 벗어나는 최소한의 확인이 이것입니다 —
**우리가 만들지 않은 실제 GPS**에 같은 규칙을 적용해도 상식적인 결과가 나오는가.

데이터: Geoff Boeing의 2014년 여름 유럽 여행 GPS 1,759개 (위경도 + 날짜, 공개 저장소).
"서로 다른 3일 이상 머문 곳 = 거점" 규칙이 실데이터에서 뭘 찾아내는지 보세요 —
**여행자가 실제로 오래 머문 도시들이 이름으로 나옵니다** (바르셀로나 21일 등).
결과가 상식과 일치하는지를 독자가 직접 검증할 수 있는 게 실데이터의 힘입니다.

※ 여기서는 eps를 500m로 씁니다 — 이 데이터의 "장소"는 주차장이 아니라 **도시 안 체류지**(숙소 일대)라
물리적 크기가 더 커서요. "eps는 그 데이터에서 한 장소의 크기에 맞춘다"는 원칙의 실례입니다.

다른 실데이터 옵션 (더 크고 무거움):
- Microsoft GeoLife — 베이징 182명 GPS 궤적, 생활권 연구 표준: https://www.microsoft.com/en-us/download/details.aspx?id=52367
- SNAP Brightkite/Gowalla — 사용자별 위치 체크인: https://snap.stanford.edu/data/loc-Brightkite.html

In [ ]:
#@title 실제 GPS 1,759개에 MASIL 규칙(반경 500m + 서로 다른 3일) 적용
import urllib.request, io, csv as _csv
from collections import Counter

URL = "https://raw.githubusercontent.com/gboeing/2014-summer-travels/master/data/summer-travel-gps-full.csv"
raw = urllib.request.urlopen(URL).read().decode()
rows = list(_csv.DictReader(io.StringIO(raw)))
real = [{"latitude": float(r["lat"]), "longitude": float(r["lon"]),
         "visit_date": r["date"].split()[0], "city": r["city"]} for r in rows]
print(f"실데이터 로드: GPS {len(real)}개 (2014-05 ~ 2014-08, 유럽)")

res = dbscan_distinct_days(real, eps_m=500, min_distinct_days=3)
labels_real = res["labels"]

groups = defaultdict(list)
for e, l in zip(real, labels_real):
    if l >= 0:
        groups[l].append(e)
hubs = sorted(groups.values(), key=lambda g: -len({e["visit_date"] for e in g}))
print(f"\n발견된 거점 {len(hubs)}곳 / 노이즈 {res['noise_count']}개 (일회성 이동 경로)")
print(f"\n{'거점(최빈 도시)':24s} {'머문 날':>6s} {'GPS 수':>6s}")
for g in hubs[:8]:
    city = Counter(e["city"] for e in g).most_common(1)[0][0]
    print(f"{city:24s} {len({e['visit_date'] for e in g}):5d}일 {len(g):6d}개")

fig, ax = plt.subplots(figsize=(9, 7))
for e, l in zip(real, labels_real):
    if l < 0:
        ax.scatter(e["longitude"], e["latitude"], marker="x", c="#bbbbbb", s=14, zorder=2)
    else:
        ax.scatter(e["longitude"], e["latitude"], c=COLORS[l % len(COLORS)], s=22, zorder=3)
for g in hubs[:6]:
    city = Counter(e["city"] for e in g).most_common(1)[0][0]
    la = sum(e["latitude"] for e in g) / len(g)
    lo = sum(e["longitude"] for e in g) / len(g)
    ax.annotate(city, (lo, la), textcoords="offset points", xytext=(6, 6), fontsize=10)
ax.set_title("실제 여행 GPS — 색 = 서로 다른 3일 이상 머문 거점 · 회색 × = 일회성 경로(노이즈)", fontsize=11)
ax.set_xlabel("경도"); ax.set_ylabel("위도"); ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()

print("\n같은 데이터에 HDBSCAN을 적용하면 (참고):")
anchor_r = (sum(e["latitude"] for e in real) / len(real), sum(e["longitude"] for e in real) / len(real))
h_labels = run_hdbscan(real, anchor_r, 5, 3)
h_hubs = len(set(l for l in h_labels if l >= 0))
h_noise = sum(1 for l in h_labels if l < 0)
print(f"HDBSCAN(5/3): 거점 {h_hubs}곳 / 노이즈 {h_noise}개 — 대륙 규모라 평면 근사 왜곡이 있는 점은 감안")